[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/08_Evaluation_and_Runtime/Evaluation_and_Runtime_Apply.ipynb)

# 1.8 Evaluation and Runtime — Apply

Hands-on exercises using `ReferenceEvaluator` and ONNX Runtime.

---

## Table of Contents

| # | Exercise | Objective |
|---|----------|-----------|
| 1 | [ReferenceEvaluator Basics](#ex-1) | Run a model and inspect results |
| 2 | [Verbose Debugging](#ex-2) | Trace data through each node |
| 3 | [Single Node Testing](#ex-3) | Evaluate operators in isolation |
| 4 | [ONNX Runtime Inference](#ex-4) | Run the same model with ORT |
| 5 | [Custom Operator](#ex-5) | Implement a Clamp op |
| 6 | [Performance Benchmark](#ex-6) | Compare runtimes on a multi-layer model |
| 7 | [Correctness Testing](#ex-7) | Systematic comparison |
| 8 | [Challenge: Debugging Pipeline](#ex-8) | Build a full debug workflow |

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import time

from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid,
    make_tensor)
from onnx.numpy_helper import from_array
from onnx.checker import check_model
from onnx.reference import ReferenceEvaluator
from onnx.reference.op_run import OpRun
import onnxruntime as ort

print('Setup complete.')

<a id='ex-1'></a>
## Exercise 1: ReferenceEvaluator Basics

Build a model for $Y = \mathrm{Relu}(XA + B)$ and run it with `ReferenceEvaluator`.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 3])
A = make_tensor_value_info('A', TensorProto.FLOAT, [3, 2])
B = make_tensor_value_info('B', TensorProto.FLOAT, [1, 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Z']),
     make_node('Relu', ['Z'], ['Y'])],
    'relu_lr', [X, A, B], [Y])
model = make_model(graph)
check_model(model)

x = np.array([[1, -1, 2], [0, 3, -2]], dtype=np.float32)
a = np.array([[0.5, -0.3], [0.2, 0.8], [-0.1, 0.4]], dtype=np.float32)
b = np.array([[0.1, -0.5]], dtype=np.float32)

ref_sess = ReferenceEvaluator(model)
result = ref_sess.run(None, {'X': x, 'A': a, 'B': b})

expected = np.maximum(0, x @ a + b)
print(f'Result:   {result[0]}')
print(f'Expected: {expected}')
print(f'Match: {np.allclose(result[0], expected)}')

<a id='ex-2'></a>
## Exercise 2: Verbose Debugging

Run the same model with `verbose=3` to trace intermediate values through every node.

In [ ]:
print('=== Step-by-step trace (verbose=3) ===')
debug_sess = ReferenceEvaluator(model, verbose=3)
_ = debug_sess.run(None, {'X': x, 'A': a, 'B': b})

<a id='ex-3'></a>
## Exercise 3: Single Node Testing

Test the `Gemm` operator (General Matrix Multiply) in isolation. `Gemm` computes:

$$Y = \alpha \cdot A \cdot B + \beta \cdot C$$

In [ ]:
gemm_node = make_node('Gemm', ['A', 'B', 'C'], ['Y'],
                       alpha=2.0, beta=0.5, transB=1)
gemm_sess = ReferenceEvaluator(gemm_node)

a = np.array([[1, 2], [3, 4]], dtype=np.float32)
b = np.array([[5, 7], [6, 8]], dtype=np.float32)
c = np.array([[1, 1], [1, 1]], dtype=np.float32)

result = gemm_sess.run(None, {'A': a, 'B': b, 'C': c})
expected = 2.0 * (a @ b.T) + 0.5 * c

print(f'Gemm result: \n{result[0]}')
print(f'Expected (2*A@B^T + 0.5*C): \n{expected}')
print(f'Match: {np.allclose(result[0], expected)}')

<a id='ex-4'></a>
## Exercise 4: ONNX Runtime Inference

Run the Relu model from Exercise 1 with ORT and verify results match.

In [ ]:
ort_sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[1, -1, 2], [0, 3, -2]], dtype=np.float32)
a_val = np.array([[0.5, -0.3], [0.2, 0.8], [-0.1, 0.4]], dtype=np.float32)
b_val = np.array([[0.1, -0.5]], dtype=np.float32)
feeds = {'X': x, 'A': a_val, 'B': b_val}

ref_out = ReferenceEvaluator(model).run(None, feeds)[0]
ort_out = ort_sess.run(None, feeds)[0]

print(f'Ref result: {ref_out}')
print(f'ORT result: {ort_out}')
print(f'Max diff: {np.abs(ref_out - ort_out).max():.2e}')
print(f'Match: {np.allclose(ref_out, ort_out)}')

<a id='ex-5'></a>
## Exercise 5: Custom Operator — Clamp

Implement a custom `Clamp` operator that clips values to $[\text{low}, \text{high}]$:

$$\text{Clamp}(x) = \min(\max(x, \text{low}), \text{high})$$

In [ ]:
class Clamp(OpRun):
    op_domain = 'my_ops'

    def _run(self, X, low=-1.0, high=1.0):
        return (np.clip(X, low, high),)

# Build model using the custom op
X = make_tensor_value_info('X', TensorProto.FLOAT, [None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('Clamp', ['X'], ['Y'], domain='my_ops',
               low=-0.5, high=0.5)],
    'clamp_model', [X], [Y])
clamp_model = make_model(graph, opset_imports=[
    make_opsetid('', 14), make_opsetid('my_ops', 1)])

sess = ReferenceEvaluator(clamp_model, new_ops=[Clamp])

x = np.array([-2, -0.3, 0, 0.7, 1.5], dtype=np.float32)
result = sess.run(None, {'X': x})

print(f'Input:    {x}')
print(f'Clamped:  {result[0]}')
print(f'Expected: {np.clip(x, -0.5, 0.5)}')
print(f'Match: {np.allclose(result[0], np.clip(x, -0.5, 0.5))}')

<a id='ex-6'></a>
## Exercise 6: Performance Benchmark

Build a 3-layer model and benchmark both runtimes.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 64])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 8])

w1 = from_array(np.random.randn(64, 32).astype(np.float32), name='W1')
b1 = from_array(np.random.randn(32).astype(np.float32), name='B1')
w2 = from_array(np.random.randn(32, 16).astype(np.float32), name='W2')
b2 = from_array(np.random.randn(16).astype(np.float32), name='B2')
w3 = from_array(np.random.randn(16, 8).astype(np.float32), name='W3')
b3 = from_array(np.random.randn(8).astype(np.float32), name='B3')

nodes = [
    make_node('MatMul', ['X', 'W1'], ['H1_pre']),
    make_node('Add', ['H1_pre', 'B1'], ['H1_add']),
    make_node('Relu', ['H1_add'], ['H1']),
    make_node('MatMul', ['H1', 'W2'], ['H2_pre']),
    make_node('Add', ['H2_pre', 'B2'], ['H2_add']),
    make_node('Relu', ['H2_add'], ['H2']),
    make_node('MatMul', ['H2', 'W3'], ['H3_pre']),
    make_node('Add', ['H3_pre', 'B3'], ['Y']),
]

graph = make_graph(nodes, 'mlp_3layer', [X], [Y],
                   initializer=[w1, b1, w2, b2, w3, b3])
mlp_model = make_model(graph)
check_model(mlp_model)

batch_sizes = [1, 10, 50, 100, 500]
ref_times = []
ort_times = []
n_runs = 30

ref_sess = ReferenceEvaluator(mlp_model)
ort_sess = ort.InferenceSession(
    mlp_model.SerializeToString(), providers=['CPUExecutionProvider'])

for bs in batch_sizes:
    x_data = np.random.randn(bs, 64).astype(np.float32)
    feeds = {'X': x_data}

    ref_sess.run(None, feeds)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        ref_sess.run(None, feeds)
    ref_t = (time.perf_counter() - t0) / n_runs * 1000

    ort_sess.run(None, feeds)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        ort_sess.run(None, feeds)
    ort_t = (time.perf_counter() - t0) / n_runs * 1000

    ref_times.append(ref_t)
    ort_times.append(ort_t)
    print(f'  batch={bs:>3d}  Ref={ref_t:.3f}ms  ORT={ort_t:.3f}ms  '
          f'Speedup={ref_t/ort_t:.1f}×')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, ref_times, 'o-', color='#E74C3C', linewidth=2,
        markersize=8, label='ReferenceEvaluator')
ax.plot(batch_sizes, ort_times, 's-', color='#3498DB', linewidth=2,
        markersize=8, label='ONNX Runtime')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('3-Layer MLP: Latency vs Batch Size', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

<a id='ex-7'></a>
## Exercise 7: Correctness Testing

Write a systematic function that compares both runtimes across random inputs.

In [ ]:
def verify_model(model_proto, input_shapes, n_tests=10, atol=1e-5):
    """Run random inputs through both runtimes and compare."""
    ref = ReferenceEvaluator(model_proto)
    ort_s = ort.InferenceSession(
        model_proto.SerializeToString(), providers=['CPUExecutionProvider'])

    max_diffs = []
    for i in range(n_tests):
        feeds = {name: np.random.randn(*shape).astype(np.float32)
                 for name, shape in input_shapes.items()}
        ref_out = ref.run(None, feeds)
        ort_out = ort_s.run(None, feeds)

        max_diff = max(np.abs(r - o).max()
                       for r, o in zip(ref_out, ort_out))
        max_diffs.append(max_diff)

    passed = all(d < atol for d in max_diffs)
    print(f'  Tests: {n_tests} | Max diff: {max(max_diffs):.2e} | '
          f'Tolerance: {atol:.0e} | {"PASS" if passed else "FAIL"}')
    return max_diffs

print('Testing 3-layer MLP:')
diffs = verify_model(mlp_model, {'X': (50, 64)}, n_tests=20)

plt.figure(figsize=(10, 4))
plt.bar(range(len(diffs)), diffs, color='#2ECC71', alpha=0.7)
plt.axhline(y=1e-5, color='red', linestyle='--', label='Tolerance (1e-5)')
plt.xlabel('Test Run', fontsize=12)
plt.ylabel('Max Absolute Difference', fontsize=12)
plt.title('Numerical Agreement: Ref vs ORT', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

<a id='ex-8'></a>
## Exercise 8 — Challenge: Full Debugging Pipeline

Build a complete model debug utility that:

1. Takes an ONNX `ModelProto`
2. Validates it with `check_model`
3. Runs it with ReferenceEvaluator in verbose mode
4. Runs it with ONNX Runtime
5. Compares outputs
6. Reports timing

Then use it to debug a model that produces `NaN` values.

In [ ]:
def debug_model(model_proto, feeds, verbose=2):
    """Full model debugging pipeline."""
    print('=' * 60)
    print('MODEL DEBUG REPORT')
    print('=' * 60)

    # Step 1: Validate
    print('\n[1] Validation...')
    try:
        check_model(model_proto)
        print('    PASS')
    except Exception as e:
        print(f'    FAIL: {e}')
        return

    # Step 2: Model summary
    g = model_proto.graph
    print(f'\n[2] Model summary')
    print(f'    Graph: {g.name}')
    print(f'    Nodes: {len(g.node)}')
    print(f'    Inputs: {[i.name for i in g.input]}')
    print(f'    Outputs: {[o.name for o in g.output]}')
    print(f'    Initializers: {len(g.initializer)}')

    # Step 3: ReferenceEvaluator with verbose
    print(f'\n[3] ReferenceEvaluator (verbose={verbose})...')
    ref = ReferenceEvaluator(model_proto, verbose=verbose)
    t0 = time.perf_counter()
    ref_out = ref.run(None, feeds)
    ref_t = (time.perf_counter() - t0) * 1000
    print(f'    Time: {ref_t:.3f}ms')

    # Step 4: ONNX Runtime
    print(f'\n[4] ONNX Runtime...')
    ort_s = ort.InferenceSession(
        model_proto.SerializeToString(), providers=['CPUExecutionProvider'])
    t0 = time.perf_counter()
    ort_out = ort_s.run(None, feeds)
    ort_t = (time.perf_counter() - t0) * 1000
    print(f'    Time: {ort_t:.3f}ms')

    # Step 5: Compare
    print(f'\n[5] Output comparison')
    for i, (r, o) in enumerate(zip(ref_out, ort_out)):
        has_nan = np.isnan(r).any() or np.isnan(o).any()
        diff = np.abs(r - o).max() if not has_nan else float('inf')
        status = 'NaN DETECTED' if has_nan else (
            'PASS' if diff < 1e-5 else 'MISMATCH')
        print(f'    Output[{i}]: shape={r.shape}, '
              f'max_diff={diff:.2e}, status={status}')

    print(f'\n[6] Performance: Ref={ref_t:.3f}ms, ORT={ort_t:.3f}ms, '
          f'Speedup={ref_t/ort_t:.1f}×')
    print('=' * 60)
    return ref_out, ort_out

# Test with the MLP model
x = np.random.randn(10, 64).astype(np.float32)
_ = debug_model(mlp_model, {'X': x}, verbose=2)

In [ ]:
# Debug a model that might produce extreme values
# (Log of values including negatives → NaN)
X_info = make_tensor_value_info('X', TensorProto.FLOAT, [None])
Y_info = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

log_graph = make_graph(
    [make_node('Log', ['X'], ['Y'])],
    'log_test', [X_info], [Y_info])
log_model = make_model(log_graph)

x_safe = np.array([1, 2, 3, 4, 5], dtype=np.float32)
x_bad = np.array([1, -2, 3, 0, 5], dtype=np.float32)

print('>>> Test with safe inputs (all positive):')
_ = debug_model(log_model, {'X': x_safe}, verbose=0)

print('\n>>> Test with bad inputs (negatives and zero):')
_ = debug_model(log_model, {'X': x_bad}, verbose=0)

### What We Learned

The debugging pipeline caught the `NaN` issue immediately. In practice, you'd add an `Abs` or `Clip` node before `Log` to prevent invalid inputs:

```
X ──► Abs ──► Log ──► Y   (safe: no negative inputs)
```

---

**Congratulations!** You've completed all 8 topics in the ONNX with Python module.